# Construcción de la tabla `master`

Este notebook parte de los CSV crudos generados por `api_nba.ipynb` y
construye la tabla `master`, con una fila por partido y todas las
*features* que consume el modelo final. El proceso se divide en cinco
bloques: descarga por temporada, validación de inactivos, concatenación,
ingeniería de *features* de equipo y agregación de *features* de jugador.

## 1. Descarga por temporada de estadísticas de equipo y jugador

Se recorren todas las temporadas (2006-07 a 2024-25) y se descargan las
seis vistas necesarias, tres a nivel de equipo (`base`, `advanced` y
`misc`) y tres a nivel de jugador. Los ficheros se guardan por
temporada en `datasets/_raw/`.

In [2]:
import pandas as pd
import numpy as np
import time
import warnings

from requests.exceptions import ReadTimeout
from nba_api.stats.endpoints import (
    teamgamelogs,
    playergamelogs,
    boxscoresummaryv2,
    boxscoretraditionalv2
)
SEASONS = ['2006-07', '2007-08', '2008-09', '2009-10', '2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']


In [2]:
for season in SEASONS:
    """
    team_base = teamgamelogs.TeamGameLogs(season_nullable=season, season_type_nullable="Regular Season", measure_type_player_game_logs_nullable="Base").get_data_frames()[0]
    team_advanced = teamgamelogs.TeamGameLogs(season_nullable=season, season_type_nullable="Regular Season", measure_type_player_game_logs_nullable="Advanced").get_data_frames()[0]
    team_misc = teamgamelogs.TeamGameLogs(season_nullable=season, season_type_nullable="Regular Season", measure_type_player_game_logs_nullable="Misc").get_data_frames()[0]
    player_base = playergamelogs.PlayerGameLogs(season_nullable=season, season_type_nullable="Regular Season", measure_type_player_game_logs_nullable="Base").get_data_frames()[0]
    player_advanced = playergamelogs.PlayerGameLogs(season_nullable=season, season_type_nullable="Regular Season", measure_type_player_game_logs_nullable="Advanced").get_data_frames()[0]
    player_misc = playergamelogs.PlayerGameLogs(season_nullable=season, season_type_nullable="Regular Season", measure_type_player_game_logs_nullable="Misc").get_data_frames()[0]
    team_base.to_csv(f'datasets/_raw/team_base_{season}.csv', index=False)
    team_advanced.to_csv(f'datasets/_raw/team_advanced_{season}.csv', index=False)
    team_misc.to_csv(f'datasets/_raw/team_misc_{season}.csv', index=False)
    player_base.to_csv(f'datasets/_raw/player_base_{season}.csv', index=False)
    player_advanced.to_csv(f'datasets/_raw/player_advanced_{season}.csv', index=False)
    player_misc.to_csv(f'datasets/_raw/player_misc_{season}.csv', index=False)
    """

## 2. Descarga de la lista de inactivos por partido

Para cada partido se consulta el *endpoint* `BoxScoreSummaryV2`, que
devuelve la lista de jugadores marcados como inactivos antes del salto
inicial. Esta información permitirá restringir la agregación de
*features* de jugador a los convocados y evitar el *leakage* de usar
directamente los que finalmente jugaron.

In [9]:
warnings.filterwarnings(
    "ignore",
    message="BoxScoreSummaryV2 has known data availability issues.*"
)

#SEASONS = ['2024-25']
SEASONS = ['2006-07', '2007-08', '2008-09', '2009-10', '2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
SEASONS = ['2023-24', '2024-25']

for season in SEASONS:

    inactive_players = []
    
    print(f'\n=== Season {season} ===')

    team_base = teamgamelogs.TeamGameLogs(
        season_nullable=season,
        season_type_nullable="Regular Season",
        measure_type_player_game_logs_nullable="Advanced" # Base, Advanced, Misc
    ).get_data_frames()[0]
    #player_base = playergamelogs.PlayerGameLogs(
    #    season_nullable=season,
    #    season_type_nullable="Regular Season",
    #    measure_type_player_game_logs_nullable="Misc" # Base, Advanced, Misc
    #).get_data_frames()[0]
    # Guardar si quieres
    team_base.to_csv(f'datasets/_raw/team_advanced_{season}.csv', index=False)
    #player_base.to_csv(f'datasets/_raw/player_base_{season}.csv', index=False)
    games = team_base['GAME_ID'].unique()

    """
    for i, game in enumerate(games):

        success = False

        # hasta 3 intentos
        for attempt in range(3):

            try:

                df = boxscoresummaryv2.BoxScoreSummaryV2(
                    game_id=game,
                    timeout=60
                ).get_data_frames()[3]

                if not df.empty and 'PLAYER_ID' in df.columns:
                    df['GAME_ID'] = game
                    inactive_players.append(df)

                success = True
                break

            except ReadTimeout:

                print(f'Timeout game {game} intento {attempt+1}')

                time.sleep(5)

            except Exception as e:

                print(f'Error game {game}: {e}')
                break

        if not success:
            print(f'No se pudo descargar {game}')

        # MUY IMPORTANTE
        time.sleep(0.5)

        if i % 25 == 0:
            print(f'{i}/{len(games)} partidos')
    # DataFrame final
    
    inactive_players = pd.concat(inactive_players, ignore_index=True)

    inactive_players.to_csv(f'datasets/inactive_players_{season}.csv',index=False)
    """


print('\nFinalizado')


=== Season 2023-24 ===

=== Season 2024-25 ===

Finalizado


## 2. Descarga de la lista de inactivos por partido

Para cada partido se consulta el *endpoint* `BoxScoreSummaryV2`, que
devuelve la lista de jugadores marcados como inactivos antes del salto
inicial. Esta información permitirá restringir la agregación de
*features* de jugador a los convocados y evitar el *leakage* de usar
directamente los que finalmente jugaron.

In [4]:
SEASONS = ['2006-07', '2007-08', '2008-09', '2009-10', '2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23']

for season in SEASONS:
    print(f'\n=== Season {season} ===')
    team_base = pd.read_csv(f'datasets/_raw/team_base_{season}.csv')
    team_advanced = pd.read_csv(f'datasets/_raw/team_advanced_{season}.csv')
    team_misc = pd.read_csv(f'datasets/_raw/team_misc_{season}.csv')
    player_base = pd.read_csv(f'datasets/_raw/player_base_{season}.csv')
    player_advanced = pd.read_csv(f'datasets/_raw/player_advanced_{season}.csv')
    player_misc = pd.read_csv(f'datasets/_raw/player_misc_{season}.csv')
    tbl = len(team_base['GAME_ID'].unique())
    tal = len(team_advanced['GAME_ID'].unique())
    tml = len(team_misc['GAME_ID'].unique())
    pbl = len(player_base['GAME_ID'].unique())
    pal = len(player_advanced['GAME_ID'].unique())
    pml = len(player_misc['GAME_ID'].unique())
    
    if tbl == tal == tml == pbl == pal == pml:
        print('Todos los partidos coinciden')
    else:
        print('Partidos no coinciden')
        print('Team Base:', set(tbl))
        print('Team Advanced:', set(tal))
        print('Team Misc:', set(tml))
        print('Player Base:', set(pbl))
        print('Player Advanced:', set(pal))
        print('Player Misc:', set(pml))

    print('---')


=== Season 2006-07 ===
Todos los partidos coinciden
---

=== Season 2007-08 ===
Todos los partidos coinciden
---

=== Season 2008-09 ===
Todos los partidos coinciden
---

=== Season 2009-10 ===
Todos los partidos coinciden
---

=== Season 2010-11 ===
Todos los partidos coinciden
---

=== Season 2011-12 ===
Todos los partidos coinciden
---

=== Season 2012-13 ===
Todos los partidos coinciden
---

=== Season 2013-14 ===
Todos los partidos coinciden
---

=== Season 2014-15 ===
Todos los partidos coinciden
---

=== Season 2015-16 ===
Todos los partidos coinciden
---

=== Season 2016-17 ===
Todos los partidos coinciden
---

=== Season 2017-18 ===
Todos los partidos coinciden
---

=== Season 2018-19 ===
Todos los partidos coinciden
---

=== Season 2019-20 ===
Todos los partidos coinciden
---

=== Season 2020-21 ===
Todos los partidos coinciden
---

=== Season 2021-22 ===
Todos los partidos coinciden
---

=== Season 2022-23 ===
Todos los partidos coinciden
---


## 4. Comparacion de inactivos

Comparación entre los jugadores marcados como inactivos y los que
realmente aparecen en los *box scores*. Sirve para confirmar que ambos
conjuntos son consistentes y que la lista de convocados puede usarse sin
riesgo de perder información.

In [19]:
warnings.filterwarnings("ignore", message="BoxScoreTraditionalV2 is deprecated and will be removed in a future version. Please use BoxScoreTraditionalV3 instead. Data is no longer being published for BoxScoreTraditionalV2 as of the 2025-26 NBA season.*")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

inactive_players = pd.read_csv('datasets/inactive_players.csv', dtype={'JERSEY_NUM': str, 'GAME_ID': str})
inactive_games = set(inactive_players['GAME_ID'].unique())

all_games = set()
for season in [s for s in SEASONS if s != '2024-25']: # No hay datos de 2024-25
    team_base = pd.read_csv(f'datasets/_raw/team_base_{season}.csv',dtype={'GAME_ID': str})
    all_games.update(team_base['GAME_ID'].unique())

missing_games = all_games - inactive_games

print(f'Partidos faltantes: {len(missing_games)}')
print(sorted(missing_games))

for game in sorted(missing_games):
    print(f'\n--- Partido {game} ---')
    df = boxscoresummaryv2.BoxScoreSummaryV2(game_id=game,timeout=10).get_data_frames()[3]
    print(len(df))
    time.sleep(5)
    df = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id=game,timeout=10).get_data_frames()[0]
    print(df['COMMENT'].unique())
    time.sleep(5)
    print("\n\n")

Partidos faltantes: 15
['0020600013', '0020600019', '0020600044', '0020600116', '0021100049', '0021100087', '0021100099', '0021100127', '0021100130', '0021100367', '0021100557', '0021100626', '0021100714', '0021100943', '0021400612']

--- Partido 0020600013 ---
0
['' "DNP - Coach's Decision" 'NWT - League Suspension']




--- Partido 0020600019 ---
0
['' "DNP - Coach's Decision"]




--- Partido 0020600044 ---
0
['']




--- Partido 0020600116 ---
0
['' "DNP - Coach's Decision"]




--- Partido 0021100049 ---
0
['' "DNP - Coach's Decision" 'DND - Sore right groin.'
 'DND - Torn right lateral meniscus.' 'DND - Sprained right foot.']




--- Partido 0021100087 ---
0
['' 'DND - LEFT ELBOW SPRAIN' "DNP - Coach's Decision"]




--- Partido 0021100099 ---
0
['' 'NWT - Family Matters' "DNP - Coach's Decision"]




--- Partido 0021100127 ---
0
['' "DNP - Coach's Decision" 'NWT - Food Poisoning']




--- Partido 0021100130 ---
0
['' "DNP - Coach's Decision" 'DND - Sore Right Ankle'
 'DND - Left

### Validación puntual sobre un partido concreto

Ejemplo mínimo con un `GAME_ID` para inspeccionar caso a caso que los
jugadores marcados como inactivos por sus equipos no aparecen en el
*box score* traditional del partido.

In [20]:
df_inactive = boxscoresummaryv2.BoxScoreSummaryV2(game_id='0021100032',timeout=10).get_data_frames()[3]
print(df_inactive)
inactive_ids = df_inactive['PLAYER_ID'].unique()
df_game = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id='0021100032',timeout=10).get_data_frames()[0]
for id in inactive_ids:
    print(f'\n--- PLAYER_ID {id} ---')
    found = df_game[df_game['PLAYER_ID'] == id]
    if not found.empty:
        print(found[['PLAYER_ID', 'COMMENT']].to_string(index=False))
    else:
        print('No encontrado')

   PLAYER_ID FIRST_NAME LAST_NAME JERSEY_NUM     TEAM_ID     TEAM_CITY  \
0       1884      Baron     Davis       85    1610612752      New York   
1       2407      Jared  Jeffries        9    1610612752      New York   
2     202697       Iman  Shumpert       21    1610612752      New York   
3     201939    Stephen     Curry       30    1610612744  Golden State   

  TEAM_NAME TEAM_ABBREVIATION  
0    Knicks               NYK  
1    Knicks               NYK  
2    Knicks               NYK  
3  Warriors               GSW  

--- PLAYER_ID 1884 ---
No encontrado

--- PLAYER_ID 2407 ---
No encontrado

--- PLAYER_ID 202697 ---
No encontrado

--- PLAYER_ID 201939 ---
No encontrado


## 5. Concatenación por temporada

Unión de los CSV por temporada en seis ficheros globales
(`*_full.csv`), uno por vista. A partir de aquí, el resto del pipeline
trabaja siempre sobre estos seis ficheros y no vuelve a la NBA API.

In [10]:
options = ["team_base", "team_advanced", "team_misc", "player_base", "player_advanced", "player_misc"]

for option in options:
    full_df = pd.concat([pd.read_csv(f'datasets/_raw/{option}_{season}.csv', dtype={'GAME_ID': str}) for season in SEASONS], ignore_index=True)
    full_df.to_csv(f'datasets/{option}_full.csv', index=False)
    print(f'{option} guardado')

team_base guardado
team_advanced guardado
team_misc guardado
player_base guardado
player_advanced guardado
player_misc guardado


In [11]:
df_team_base = pd.read_csv('datasets/team_base_full.csv', dtype={'GAME_ID': str})
df_team_advanced = pd.read_csv('datasets/team_advanced_full.csv', dtype={'GAME_ID': str})
df_team_misc = pd.read_csv('datasets/team_misc_full.csv', dtype={'GAME_ID': str})
df_player_base = pd.read_csv('datasets/player_base_full.csv', dtype={'GAME_ID': str})
df_player_advanced = pd.read_csv('datasets/player_advanced_full.csv', dtype={'GAME_ID': str})
df_player_misc = pd.read_csv('datasets/player_misc_full.csv', dtype={'GAME_ID': str})
print('DataFrames cargados')
print('Team Base:', df_team_base.shape)
print('Team Advanced:', df_team_advanced.shape)
print('Team Misc:', df_team_misc.shape)
print('Player Base:', df_player_base.shape)
print('Player Advanced:', df_player_advanced.shape)
print('Player Misc:', df_player_misc.shape)
print(f'Partidos únicos en Team Base: {len(df_team_base["GAME_ID"].unique())}')
print(f'Partidos únicos en Team Advanced: {len(df_team_advanced["GAME_ID"].unique())}')
print(f'Partidos únicos en Team Misc: {len(df_team_misc["GAME_ID"].unique())}')
print(f'Partidos únicos en Player Base: {len(df_player_base["GAME_ID"].unique())}')
print(f'Partidos únicos en Player Advanced: {len(df_player_advanced["GAME_ID"].unique())}')
print(f'Partidos únicos en Player Misc: {len(df_player_misc["GAME_ID"].unique())}')

DataFrames cargados
Team Base: (4920, 57)
Team Advanced: (4920, 49)
Team Misc: (4920, 31)
Player Base: (52707, 70)
Player Advanced: (52707, 82)
Player Misc: (52707, 46)
Partidos únicos en Team Base: 2460
Partidos únicos en Team Advanced: 2460
Partidos únicos en Team Misc: 2460
Partidos únicos en Player Base: 2460
Partidos únicos en Player Advanced: 2460
Partidos únicos en Player Misc: 2460


In [12]:
print("Columnas de Team Base:", df_team_base.columns.tolist())
print("Columnas de Team Advanced:", df_team_advanced.columns.tolist())
print("Columnas de Team Misc:", df_team_misc.columns.tolist())
print("Columnas de Player Base:", df_player_base.columns.tolist())
print("Columnas de Player Advanced:", df_player_advanced.columns.tolist())
print("Columnas de Player Misc:", df_player_misc.columns.tolist())

Columnas de Team Base: ['SEASON_YEAR', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'AVAILABLE_FLAG']
Columnas de Team Advanced: ['SEASON_YEAR', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'E_OFF_RATING', 'OFF_RATING', 'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'AST_PCT', 'AST_TO', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT', 'EFG_PCT', 'TS_PCT', 'E_PACE', 'PACE', 'PAC

### Ingeniería de *features* de equipo

Función `add_rolling_features` que, para cada columna estadística,
calcula la media móvil de los últimos N partidos y la media acumulada
de la temporada actual. La aplicación de `shift(1)` garantiza que el
partido a predecir nunca entra en su propia media.

In [13]:
import pandas as pd
pd.set_option('display.max_columns', None)

# ---------- Helpers ----------
def add_rolling_features(df, cols=None, windows=(5, 10), season=True,
                         group=('SEASON_YEAR', 'TEAM_ID'), date='GAME_DATE'):
    """Por columna: media de los N partidos previos y acumulado de temporada.
       shift(1) -> el partido actual nunca entra en su media (sin leakage)."""
    df = df.sort_values([*group, date]).copy()
    if cols is None:                                  # por defecto, todas las stats numericas
        skip = {*group, 'TEAM_ID', 'GAME_ID', 'MIN'}
        cols = [c for c in df.select_dtypes('number').columns if c not in skip]
    g = df.groupby(list(group))
    new = {}
    for c in cols:
        for w in windows:
            new[f'{c}_L{w}'] = g[c].transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
        if season:
            new[f'{c}_season'] = g[c].transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
    return pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)

def signed_streak(wl):
    """WWLW -> 1, 2, -1, 1."""
    out, cur = [], 0
    for r in wl:
        cur = (cur + 1 if cur > 0 else 1) if r == 'W' else (cur - 1 if cur < 0 else -1)
        out.append(cur)
    return out

# ---------- 1. Unir base + advanced + misc ----------
shared_meta = ['SEASON_YEAR', 'GAME_DATE', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'MATCHUP', 'WL', 'MIN']

def clean(d, base=False):
    drop = [c for c in d.columns if c.endswith('_RANK')] + ['AVAILABLE_FLAG']
    if not base:
        drop += shared_meta                            # los metadatos solo en base
    return d.drop(columns=drop, errors='ignore')

df = clean(df_team_base, base=True) \
        .merge(clean(df_team_advanced), on=['GAME_ID', 'TEAM_ID']) \
        .merge(clean(df_team_misc),     on=['GAME_ID', 'TEAM_ID'])
df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])

# ---------- 2. Four Factors: solo falta la tasa de tiros libres ----------
df['ff_ftr'] = df['FTA'] / df['FGA']                       # el 4o factor que no esta en Advanced

# Variable combinada con los pesos estandar (40/25/20/15).
# Estandarizo cada factor porque van en escalas distintas; el de perdidas resta (menos es mejor).
z = lambda s: (s - s.mean()) / s.std()
df['four_factors'] = ( 0.40 * z(df['EFG_PCT'])
                     - 0.25 * z(df['TM_TOV_PCT'])
                     + 0.20 * z(df['OREB_PCT'])
                     + 0.15 * z(df['ff_ftr']) )                                 # tiros libres por tiro

# ---------- 3. Rolling + temporada (incluye los Four Factors) ----------
df = add_rolling_features(df, windows=(5, 10))

# ---------- 4. Descanso y racha ----------
df['days_rest'] = df.groupby(['SEASON_YEAR', 'TEAM_ID'])['GAME_DATE'].diff().dt.days
df['streak'] = df.groupby(['SEASON_YEAR', 'TEAM_ID'])['WL'].transform(
    lambda s: pd.Series(signed_streak(s.values), index=s.index))
df['streak'] = df.groupby(['SEASON_YEAR', 'TEAM_ID'])['streak'].shift(1)

# ---------- 5. Pivote a una fila por partido ----------
keys = ['GAME_ID', 'GAME_DATE', 'SEASON_YEAR']
is_home = df['MATCHUP'].str.contains('vs.', regex=False)
home = df[is_home].rename(columns=lambda c: c if c in keys else f'home_{c}')
away = df[~is_home].rename(columns=lambda c: c if c in keys else f'away_{c}')
master = home.merge(away, on=keys)

# ---------- 6. Target, MIN, descanso y racha (home / away / diff) ----------
master['home_team_won'] = (master['home_WL'] == 'W').astype(int)
master['MIN'] = master['home_MIN']
for col in ['days_rest', 'streak']:
    master[f'{col}_home'] = master[f'home_{col}']
    master[f'{col}_away'] = master[f'away_{col}']
    master[f'{col}_diff'] = master[f'home_{col}'] - master[f'away_{col}']

# ---------- 7. Resta home - away de cada feature rolling/temporada ----------
roll = [c[5:] for c in master.columns
        if c.startswith('home_') and ('_L5' in c or '_L10' in c or c.endswith('_season'))]
master = pd.concat([master,
    pd.DataFrame({f'diff_{c}': master[f'home_{c}'] - master[f'away_{c}'] for c in roll})], axis=1)

# Me quedo con claves, ids, target, descanso/racha y las diff_
keep_ids = ('TEAM_ID', 'TEAM_ABBREVIATION')
master = master.drop(columns=[c for c in master.columns
    if c.startswith(('home_', 'away_')) and c != 'home_team_won' and c[5:] not in keep_ids])

print('master:', master.shape)

master: (2455, 168)


### Ingeniería de *features* de jugador

Función `player_metric_diff` que pondera cada métrica por minutos
jugados, agrega los valores por partido restringiendo a los jugadores
convocados y calcula la diferencia entre local y visitante. Es la
principal aportación técnica del trabajo.

In [14]:
import pandas as pd
# add_rolling_features ya esta definida arriba

# Helper: metrica de jugador ponderada por minutos -> diff home/away por partido
def player_metric_diff(pdf, metric, name):
    pdf = pdf.copy()
    pdf['w'] = pdf[metric] * (pdf['MIN'] / 48)          # peso (metrica)*(min/48)
    pdf = add_rolling_features(pdf, cols=['w'], windows=(5, 10),
                               group=('SEASON_YEAR', 'PLAYER_ID'))
    pdf['is_home'] = pdf['MATCHUP'].str.contains('vs.', regex=False)
    played = pdf[pdf['MIN'] > 0]
    wins = ['w_L5', 'w_L10', 'w_season']
    agg = played.groupby(['GAME_ID', 'is_home'])[wins].sum().unstack('is_home')
    out = pd.DataFrame(index=agg.index)
    for w in wins:
        suf = w.split('_', 1)[1]                          # L5 / L10 / season
        out[f'diff_players_{name}_{suf}'] = agg[(w, True)] - agg[(w, False)]
    out = out.reset_index()
    out['GAME_ID'] = out['GAME_ID'].astype(str).str.zfill(10)
    return out

# Cargo las dos tablas de jugadores
def load_players(path):
    p = pd.read_csv(path, dtype={'GAME_ID': str})
    p['GAME_DATE'] = pd.to_datetime(p['GAME_DATE'])
    p['MIN'] = pd.to_numeric(p['MIN'], errors='coerce')
    return p

pb = load_players('datasets/player_base_full.csv')        # tiene PLUS_MINUS
pa = load_players('datasets/player_advanced_full.csv')    # tiene PIE y NET_RATING

# Las 3 metricas de jugador: +/-, PIE y NET_RATING
master['GAME_ID'] = master['GAME_ID'].astype(str).str.zfill(10)
for pdf, metric, name in [(pb, 'PLUS_MINUS', 'pm'),
                          (pa, 'PIE',        'pie'),
                          (pa, 'NET_RATING', 'netrtg')]:
    master = master.merge(player_metric_diff(pdf, metric, name), on='GAME_ID', how='left')

print("master:", master.shape)

master: (2455, 177)


### Guardado de la tabla `master`

Se ordena por fecha y GAME_ID y se guarda como `master_v1.csv`. La
versión definitiva con `bubble` e `importance` se construye en el
bloque siguiente.

In [15]:
master.sort_values(['GAME_DATE', 'GAME_ID'], inplace=True)
master.to_csv('datasets/master_v1.csv', index=False)

## 6. Construcción de la tabla `master` (versión final)

Versión definitiva de la tabla `master` con dos novedades importantes
sobre la versión previa. Primero, el *season prior* solo se aplica a
partir del partido 10 de cada temporada, para no arrastrar sesgo desde
el `cold-start`. Segundo, se incorporan las *features* de contexto
`bubble` (indicador de burbuja COVID) e `importance` (relevancia del
partido en la lucha por playoffs).

In [16]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

# ============ Cargas ============
df_team_base     = pd.read_csv('datasets/team_base_full.csv',     dtype={'GAME_ID': str})
df_team_advanced = pd.read_csv('datasets/team_advanced_full.csv', dtype={'GAME_ID': str})
df_team_misc     = pd.read_csv('datasets/team_misc_full.csv',     dtype={'GAME_ID': str})

# ============ Helpers ============
def add_rolling(df, cols, windows, group, date='GAME_DATE'):
    """Media de los N partidos previos (shift(1) -> sin leakage)."""
    df = df.sort_values([*group, date]).copy()
    g = df.groupby(list(group))
    new = {f'{c}_L{w}': g[c].transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
           for w in windows for c in cols}
    return pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)

def add_season_prior(df, cols, group, season='SEASON_YEAR', date='GAME_DATE', min_games=10):
    """{c}_season: primeros `min_games` partidos -> media de la temporada ANTERIOR del grupo;
       desde ahi -> expanding actual (shifteada)."""
    g_season = list(group) + [season]
    df = df.sort_values(g_season + [date]).copy()
    df['_n'] = df.groupby(g_season).cumcount()
    avg = df.groupby(g_season)[list(cols)].mean().reset_index().sort_values(g_season)
    for c in cols:
        avg[f'_p_{c}'] = avg.groupby(list(group))[c].shift(1)
    df = df.merge(avg[g_season + [f'_p_{c}' for c in cols]], on=g_season, how='left')
    new = {}
    for c in cols:
        cur = df.groupby(g_season)[c].transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
        new[f'{c}_season'] = np.where(df['_n'] < min_games, df[f'_p_{c}'], cur)
    df = df.drop(columns=['_n'] + [f'_p_{c}' for c in cols])
    return pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)

def signed_streak(wl):
    """WWLW -> 1, 2, -1, 1."""
    out, cur = [], 0
    for r in wl:
        cur = (cur + 1 if cur > 0 else 1) if r == 'W' else (cur - 1 if cur < 0 else -1)
        out.append(cur)
    return out

def add_team_importance(df_team):
    """
    Para cada equipo-partido (sin leakage): game_importance = 1 si el cutoff de playoffs
    (~51.2% wins, ajustado a la longitud de temporada) sigue alcanzable desde el record actual;
    decae si el equipo ya esta clinched o eliminado.
    """
    df = df_team.sort_values(['SEASON_YEAR','TEAM_ID','GAME_DATE']).copy()
    df['_w'] = (df['WL'] == 'W').astype(int)
    g = df.groupby(['SEASON_YEAR','TEAM_ID'])
    df['wins_b']     = g['_w'].cumsum().shift(1).fillna(0)
    df['games_b']    = g.cumcount()
    df['total_g']    = g['_w'].transform('size')
    df['games_left'] = df['total_g'] - df['games_b']
    cutoff = df['total_g'] * 0.512                          # escala con lockouts/COVID
    min_w, max_w = df['wins_b'], df['wins_b'] + df['games_left']
    dist = np.maximum(0, np.maximum(cutoff - max_w, min_w - cutoff))
    df['game_importance'] = 1 / (1 + dist / 3)
    return df.drop(columns='_w')

# ============ 1. Unir base + advanced + misc ============
shared_meta = ['SEASON_YEAR','GAME_DATE','TEAM_ABBREVIATION','TEAM_NAME','MATCHUP','WL','MIN']
def clean(d, base=False):
    drop = [c for c in d.columns if c.endswith('_RANK')] + ['AVAILABLE_FLAG']
    if not base: drop += shared_meta
    return d.drop(columns=drop, errors='ignore')

df = (clean(df_team_base, base=True)
      .merge(clean(df_team_advanced), on=['GAME_ID','TEAM_ID'])
      .merge(clean(df_team_misc),     on=['GAME_ID','TEAM_ID']))
df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])

# ============ 2. Four Factors + variable combinada ============
df['ff_ftr'] = df['FTA'] / df['FGA']
z = lambda s: (s - s.mean()) / s.std()
df['four_factors'] = (0.40*z(df['EFG_PCT']) - 0.25*z(df['TM_TOV_PCT'])
                    + 0.20*z(df['OREB_PCT']) + 0.15*z(df['ff_ftr']))

# ============ 3. Rolling L5/L10 + season-con-prior (equipos resetean por temporada) ============
stat_cols = [c for c in df.select_dtypes('number').columns if c not in {'TEAM_ID','GAME_ID','MIN'}]
df = add_rolling(df, cols=stat_cols, windows=(5, 10), group=('SEASON_YEAR','TEAM_ID'))
df = add_season_prior(df, cols=stat_cols, group=('TEAM_ID',))

# ============ 4. Descanso, racha e IMPORTANCIA del partido ============
df['days_rest'] = df.groupby(['SEASON_YEAR','TEAM_ID'])['GAME_DATE'].diff().dt.days
df['streak'] = df.groupby(['SEASON_YEAR','TEAM_ID'])['WL'].transform(
    lambda s: pd.Series(signed_streak(s.values), index=s.index))
df['streak'] = df.groupby(['SEASON_YEAR','TEAM_ID'])['streak'].shift(1)
df = add_team_importance(df)

# ============ 5. Pivote a una fila por partido ============
keys = ['GAME_ID','GAME_DATE','SEASON_YEAR']
is_home = df['MATCHUP'].str.contains('vs.', regex=False)
home = df[is_home].rename(columns=lambda c: c if c in keys else f'home_{c}')
away = df[~is_home].rename(columns=lambda c: c if c in keys else f'away_{c}')
master = home.merge(away, on=keys)

# ============ 6. Target, MIN, BUBBLE, descanso/racha, IMPORTANCIA ============
master['home_team_won'] = (master['home_WL'] == 'W').astype(int)
master['MIN'] = master['home_MIN']
master['bubble'] = ((master['GAME_DATE'] >= '2020-07-30') &
                    (master['GAME_DATE'] <= '2020-10-11')).astype(int)
for col in ['days_rest','streak']:
    master[f'{col}_home'] = master[f'home_{col}']
    master[f'{col}_away'] = master[f'away_{col}']
    master[f'{col}_diff'] = master[f'home_{col}'] - master[f'away_{col}']

# Importancia: home, away, max (el partido importa si CUALQUIER equipo tiene en juego), diff
master['importance_home'] = master['home_game_importance']
master['importance_away'] = master['away_game_importance']
master['importance_max']  = master[['importance_home','importance_away']].max(axis=1)
master['importance_diff'] = master['importance_home'] - master['importance_away']

# ============ 7. Diffs home-away de cada feature rolling/season + limpieza final ============
roll = [c[5:] for c in master.columns
        if c.startswith('home_') and c.endswith(('_L5','_L10','_season'))]
master = pd.concat([master,
    pd.DataFrame({f'diff_{c}': master[f'home_{c}'] - master[f'away_{c}'] for c in roll})], axis=1)
keep_ids = ('TEAM_ID','TEAM_ABBREVIATION')
master = master.drop(columns=[c for c in master.columns
    if c.startswith(('home_','away_')) and c != 'home_team_won' and c[5:] not in keep_ids])

# ============ 8. Jugadores (pm, pie, netrtg) — cruzan temporadas ============
def load_players(path):
    p = pd.read_csv(path, dtype={'GAME_ID': str})
    p['GAME_DATE'] = pd.to_datetime(p['GAME_DATE'])
    p['MIN'] = pd.to_numeric(p['MIN'], errors='coerce')
    return p

def player_metric_diff(pdf, metric, name):
    pdf = pdf.copy()
    pdf['w'] = pdf[metric] * (pdf['MIN'] / 48)
    pdf = add_rolling(pdf, cols=['w'], windows=(5, 10), group=('PLAYER_ID',))
    pdf = add_season_prior(pdf, cols=['w'], group=('PLAYER_ID',))
    pdf['is_home'] = pdf['MATCHUP'].str.contains('vs.', regex=False)
    played = pdf[pdf['MIN'] > 0]
    wins = ['w_L5','w_L10','w_season']
    agg = played.groupby(['GAME_ID','is_home'])[wins].sum().unstack('is_home')
    out = pd.DataFrame({f'diff_players_{name}_{w.split("_",1)[1]}': agg[(w, True)] - agg[(w, False)]
                        for w in wins}).reset_index()
    out['GAME_ID'] = out['GAME_ID'].astype(str).str.zfill(10)
    return out

pb = load_players('datasets/player_base_full.csv')
pa = load_players('datasets/player_advanced_full.csv')

master['GAME_ID'] = master['GAME_ID'].astype(str).str.zfill(10)
for pdf, metric, name in [(pb, 'PLUS_MINUS', 'pm'),
                          (pa, 'PIE',        'pie'),
                          (pa, 'NET_RATING', 'netrtg')]:
    master = master.merge(player_metric_diff(pdf, metric, name), on='GAME_ID', how='left')

master = master.sort_values(['GAME_DATE','GAME_ID']).reset_index(drop=True)
master.to_csv('datasets/master_v2.csv', index=False)
print('master_v2:', master.shape)
print(f'Porcentaje victorias locales: {master["home_team_won"].mean():.2%}')

master_v2: (2455, 182)
Porcentaje victorias locales: 54.38%
